In [1]:
# 06c-1. xgboost 패키지 확인

import xgboost as xgb

from xgboost import XGBRegressor

print(
    "xgboost version:",
    xgb.__version__
)

xgboost version: 3.2.0


In [2]:
# 06c-2. 기본 설정

from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from sklearn.model_selection import ParameterSampler
from sklearn.dummy import DummyRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)


PROJECT_ROOT = Path(
    r"C:\code\portfolio_optimization"
)

SUPERVISED_DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "features"
    / "common"
    / "supervised_dataset.parquet"
)


print(
    "dataset exists:",
    SUPERVISED_DATASET_PATH.exists()
)

dataset exists: True


In [3]:
# 06c-3. supervised dataset 불러오기

supervised_dataset = (
    pq.read_table(
        SUPERVISED_DATASET_PATH
    )
    .to_pandas()
    .sort_values(
        [
            "signal_date",
            "ticker"
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "shape:",
    supervised_dataset.shape
)

print(
    "signals:",
    supervised_dataset[
        "signal_date"
    ].nunique()
)

print(
    "start:",
    supervised_dataset[
        "signal_date"
    ].min()
)

print(
    "end:",
    supervised_dataset[
        "signal_date"
    ].max()
)

shape: (21779, 39)
signals: 436
start: 2018-05-04 00:00:00
end: 2026-09-04 00:00:00


In [4]:
# 06c-4. model feature 설정

ASSET_FEATURES = [
    "return_1d",
    "return_5d",
    "momentum_20d",
    "momentum_60d",
    "volatility_20d",
    "drawdown_20d",
    "trading_value_ma20",
    "trading_value_ratio_20d",
    "log_market_cap"
]

MARKET_FEATURES = [
    "market_return_1d",
    "market_return_5d",
    "market_return_20d",
    "market_volatility_20d",
    "market_drawdown",
    "volume_change_1d",
    "trading_value_change_1d",
    "market_trading_value_ratio_20d"
]

MACRO_FEATURES = [
    "base_rate",
    "usdkrw",
    "bond3y",
    "usdkrw_return_1d",
    "usdkrw_return_5d",
    "usdkrw_return_20d",
    "bond3y_change_1d",
    "bond3y_change_5d",
    "bond3y_change_20d",
    "base_rate_change",
    "rate_spread_3y"
]


MODEL_FEATURES = (
    ASSET_FEATURES
    + MARKET_FEATURES
    + MACRO_FEATURES
)

TARGET = "target_return"


print(
    "feature count:",
    len(MODEL_FEATURES)
)

feature count: 28


In [5]:
# 06c-5. cross-sectional ic 함수

def calculate_ic(
    data,
    prediction_column,
    target_column
):

    ic_values = []

    for _, group in data.groupby(
        "signal_date"
    ):

        if len(group) < 2:
            continue

        if (
            group[prediction_column].nunique() < 2
            or
            group[target_column].nunique() < 2
        ):
            continue

        ic = (
            group[
                prediction_column
            ]
            .rank()
            .corr(
                group[
                    target_column
                ]
                .rank()
            )
        )

        if pd.notna(ic):
            ic_values.append(ic)

    return np.array(
        ic_values
    )

In [6]:
# 06c-6. walk-forward 설정

TRAIN_YEARS = 3
VALIDATION_MONTHS = 6
TEST_MONTHS = 6
STEP_MONTHS = 6


signal_start = (
    supervised_dataset[
        "signal_date"
    ].min()
)

signal_end = (
    supervised_dataset[
        "signal_date"
    ].max()
)


folds = []

validation_start = (
    signal_start
    + pd.DateOffset(
        years=TRAIN_YEARS
    )
)

fold_id = 1


while True:

    test_start = (
        validation_start
        + pd.DateOffset(
            months=VALIDATION_MONTHS
        )
    )

    test_end = (
        test_start
        + pd.DateOffset(
            months=TEST_MONTHS
        )
    )

    if test_end > signal_end:
        break

    folds.append(
        {
            "fold": fold_id,
            "train_start": signal_start,
            "train_end": validation_start,
            "validation_start": validation_start,
            "validation_end": test_start,
            "test_start": test_start,
            "test_end": test_end
        }
    )

    validation_start = (
        validation_start
        + pd.DateOffset(
            months=STEP_MONTHS
        )
    )

    fold_id += 1


fold_table = pd.DataFrame(
    folds
)

print(
    fold_table.to_string(
        index=False
    )
)

 fold train_start  train_end validation_start validation_end test_start   test_end
    1  2018-05-04 2021-05-04       2021-05-04     2021-11-04 2021-11-04 2022-05-04
    2  2018-05-04 2021-11-04       2021-11-04     2022-05-04 2022-05-04 2022-11-04
    3  2018-05-04 2022-05-04       2022-05-04     2022-11-04 2022-11-04 2023-05-04
    4  2018-05-04 2022-11-04       2022-11-04     2023-05-04 2023-05-04 2023-11-04
    5  2018-05-04 2023-05-04       2023-05-04     2023-11-04 2023-11-04 2024-05-04
    6  2018-05-04 2023-11-04       2023-11-04     2024-05-04 2024-05-04 2024-11-04
    7  2018-05-04 2024-05-04       2024-05-04     2024-11-04 2024-11-04 2025-05-04
    8  2018-05-04 2024-11-04       2024-11-04     2025-05-04 2025-05-04 2025-11-04
    9  2018-05-04 2025-05-04       2025-05-04     2025-11-04 2025-11-04 2026-05-04


XGBoost 공식 문서 기준으로
learning_rate는 각 boosting step의 보정량을 축소하는 값이고,
max_depth는 tree 복잡도를 제어함.
min_child_weight가 커질수록 split이 보수적이 되고,
subsample은 매 boosting iteration에 사용할 row 비율,
colsample_bytree는 tree마다 사용할 feature 비율이다. 
reg_alpha는 L1, reg_lambda는 L2 regularization이다.

In [7]:
# 06c-7. xgboost search space 고정

XGB_PARAM_SPACE = {
    "max_depth": [
        2,
        3,
        4,
        5,
        6
    ],
    "learning_rate": [
        0.005,
        0.01,
        0.03,
        0.05,
        0.1
    ],
    "min_child_weight": [
        1,
        5,
        10,
        20,
        50
    ],
    "subsample": [
        0.7,
        0.85,
        1.0
    ],
    "colsample_bytree": [
        0.6,
        0.8,
        1.0
    ],
    "reg_alpha": [
        0.0,
        0.01,
        0.1
    ],
    "reg_lambda": [
        1.0,
        10.0
    ]
}

In [8]:
# 06c-8. xgboost 후보 sampling

XGB_CANDIDATES = list(
    ParameterSampler(
        XGB_PARAM_SPACE,
        n_iter=36,
        random_state=42
    )
)


print(
    "candidates:",
    len(XGB_CANDIDATES)
)

print(
    XGB_CANDIDATES[:5]
)

candidates: 36
[{'subsample': 1.0, 'reg_lambda': 1.0, 'reg_alpha': 0.1, 'min_child_weight': 10, 'max_depth': 6, 'learning_rate': 0.01, 'colsample_bytree': 0.6}, {'subsample': 1.0, 'reg_lambda': 1.0, 'reg_alpha': 0.01, 'min_child_weight': 50, 'max_depth': 6, 'learning_rate': 0.01, 'colsample_bytree': 1.0}, {'subsample': 0.7, 'reg_lambda': 1.0, 'reg_alpha': 0.01, 'min_child_weight': 1, 'max_depth': 5, 'learning_rate': 0.01, 'colsample_bytree': 1.0}, {'subsample': 0.85, 'reg_lambda': 1.0, 'reg_alpha': 0.01, 'min_child_weight': 20, 'max_depth': 4, 'learning_rate': 0.01, 'colsample_bytree': 1.0}, {'subsample': 0.85, 'reg_lambda': 10.0, 'reg_alpha': 0.01, 'min_child_weight': 50, 'max_depth': 3, 'learning_rate': 0.05, 'colsample_bytree': 0.8}]


n_estimators = 3000
early_stopping_rounds = 100
으로 상한을 넉넉히 두고 validation이 학습을 멈추게 함. 

In [9]:
# 06c-9. fold 1 데이터 설정

fold = folds[0]


train_mask = (
    (
        supervised_dataset[
            "signal_date"
        ]
        < fold["train_end"]
    )
    &
    (
        supervised_dataset[
            "next_execution_date"
        ]
        <= fold["train_end"]
    )
)


validation_mask = (
    (
        supervised_dataset[
            "signal_date"
        ]
        >= fold["validation_start"]
    )
    &
    (
        supervised_dataset[
            "signal_date"
        ]
        < fold["validation_end"]
    )
    &
    (
        supervised_dataset[
            "next_execution_date"
        ]
        <= fold["validation_end"]
    )
)


train_df = (
    supervised_dataset.loc[
        train_mask
    ]
    .copy()
)

validation_df = (
    supervised_dataset.loc[
        validation_mask
    ]
    .copy()
)


print(
    "train:",
    train_df.shape
)

print(
    "validation:",
    validation_df.shape
)

train: (7791, 39)
validation: (1248, 39)


In [10]:
# 06c-10. xgboost validation 함수

def evaluate_xgb_params(
    train_df,
    validation_df
):

    X_train = train_df[
        MODEL_FEATURES
    ]

    y_train = train_df[
        TARGET
    ]

    X_validation = validation_df[
        MODEL_FEATURES
    ]

    y_validation = validation_df[
        TARGET
    ]


    results = []


    for params in XGB_CANDIDATES:

        model = XGBRegressor(
            n_estimators=3000,
            max_depth=params[
                "max_depth"
            ],
            learning_rate=params[
                "learning_rate"
            ],
            min_child_weight=params[
                "min_child_weight"
            ],
            subsample=params[
                "subsample"
            ],
            colsample_bytree=params[
                "colsample_bytree"
            ],
            reg_alpha=params[
                "reg_alpha"
            ],
            reg_lambda=params[
                "reg_lambda"
            ],
            objective="reg:squarederror",
            eval_metric="rmse",
            early_stopping_rounds=100,
            tree_method="hist",
            random_state=42,
            n_jobs=-1
        )


        model.fit(
            X_train,
            y_train,
            eval_set=[
                (
                    X_validation,
                    y_validation
                )
            ],
            verbose=False
        )


        prediction = model.predict(
            X_validation
        )


        result_df = (
            validation_df[
                [
                    "signal_date",
                    "ticker",
                    TARGET
                ]
            ]
            .copy()
        )

        result_df[
            "prediction"
        ] = prediction


        rmse = np.sqrt(
            mean_squared_error(
                y_validation,
                prediction
            )
        )

        mae = mean_absolute_error(
            y_validation,
            prediction
        )

        r2 = r2_score(
            y_validation,
            prediction
        )


        ic_values = calculate_ic(
            result_df,
            "prediction",
            TARGET
        )


        results.append(
            {
                **params,
                "best_iteration":
                    model.best_iteration,
                "rmse":
                    rmse,
                "mae":
                    mae,
                "r2":
                    r2,
                "mean_ic": (
                    ic_values.mean()
                    if len(ic_values) > 0
                    else np.nan
                ),
                "median_ic": (
                    np.median(
                        ic_values
                    )
                    if len(ic_values) > 0
                    else np.nan
                ),
                "ic_signals":
                    len(ic_values)
            }
        )


    return pd.DataFrame(
        results
    )

In [11]:
# 06c-11. fold 1 validation

xgb_validation_results = (
    evaluate_xgb_params(
        train_df,
        validation_df
    )
)


print(
    xgb_validation_results
    .sort_values(
        [
            "rmse",
            "mean_ic"
        ],
        ascending=[
            True,
            False
        ]
    )
    .head(15)
    .to_string(
        index=False
    )
)

 subsample  reg_lambda  reg_alpha  min_child_weight  max_depth  learning_rate  colsample_bytree  best_iteration     rmse      mae       r2   mean_ic  median_ic  ic_signals
      1.00         1.0       0.00                 1          4          0.030               0.6             125 0.064994 0.043999 0.016081 -0.007541  -0.019564          25
      0.85         1.0       0.10                20          2          0.100               0.8             138 0.065061 0.044018 0.014056  0.016109  -0.000640          25
      1.00         1.0       0.10                 1          5          0.030               0.6             120 0.065238 0.044256 0.008704  0.044747   0.026948          25
      0.85         1.0       0.00                 1          6          0.010               1.0             136 0.065297 0.044151 0.006913  0.075226   0.103007          25
      0.70         1.0       0.01                 1          5          0.010               1.0              38 0.065308 0.044021 0.006556  

In [12]:
# 06c-12. xgboost 최종 local search

from sklearn.model_selection import ParameterGrid


XGB_FINAL_PARAM_SPACE = {
    "max_depth": [
        3,
        4,
        5
    ],
    "learning_rate": [
        0.01,
        0.03,
        0.05
    ],
    "min_child_weight": [
        0,
        1,
        5
    ],
    "colsample_bytree": [
        0.4,
        0.6,
        0.8
    ],
    "reg_lambda": [
        0.1,
        1.0,
        10.0
    ]
}


XGB_FINAL_CANDIDATES = list(
    ParameterGrid(
        XGB_FINAL_PARAM_SPACE
    )
)


print(
    "candidates:",
    len(XGB_FINAL_CANDIDATES)
)

candidates: 243


subsample=1: 모든 train row 사용, 더 크게 갈 수 없음
reg_alpha=0: L1 penalty 없음, 더 작게 갈 수 없음

min_child_weight=0을 추가하는 이유는 기존 best 1이 lower boundary였고 공식적으로 0 이상이 허용되기 때문. 값이 커질수록 split이 더 보수적으로.

colsample_bytree=.4를 추가하는 이유도 기존 best .6이 lower boundary라서 feature subsampling을 조금 더 강하게 했을 때 좋아지는지 한 번 확인하려고 하는 것. 공식 범위는 (0,1]

In [13]:
# 06c-13. xgboost 최종 validation 함수

def evaluate_final_xgb_params(
    train_df,
    validation_df
):

    X_train = train_df[
        MODEL_FEATURES
    ]

    y_train = train_df[
        TARGET
    ]

    X_validation = validation_df[
        MODEL_FEATURES
    ]

    y_validation = validation_df[
        TARGET
    ]


    results = []


    for params in XGB_FINAL_CANDIDATES:

        model = XGBRegressor(
            n_estimators=3000,
            max_depth=params[
                "max_depth"
            ],
            learning_rate=params[
                "learning_rate"
            ],
            min_child_weight=params[
                "min_child_weight"
            ],
            subsample=1.0,
            colsample_bytree=params[
                "colsample_bytree"
            ],
            reg_alpha=0.0,
            reg_lambda=params[
                "reg_lambda"
            ],
            objective="reg:squarederror",
            eval_metric="rmse",
            early_stopping_rounds=100,
            tree_method="hist",
            random_state=42,
            n_jobs=-1
        )


        model.fit(
            X_train,
            y_train,
            eval_set=[
                (
                    X_validation,
                    y_validation
                )
            ],
            verbose=False
        )


        prediction = model.predict(
            X_validation
        )


        result_df = (
            validation_df[
                [
                    "signal_date",
                    "ticker",
                    TARGET
                ]
            ]
            .copy()
        )

        result_df[
            "prediction"
        ] = prediction


        rmse = np.sqrt(
            mean_squared_error(
                y_validation,
                prediction
            )
        )

        mae = mean_absolute_error(
            y_validation,
            prediction
        )

        r2 = r2_score(
            y_validation,
            prediction
        )


        ic_values = calculate_ic(
            result_df,
            "prediction",
            TARGET
        )


        results.append(
            {
                **params,
                "subsample": 1.0,
                "reg_alpha": 0.0,
                "best_iteration":
                    model.best_iteration,
                "rmse": rmse,
                "mae": mae,
                "r2": r2,
                "mean_ic": (
                    ic_values.mean()
                    if len(ic_values) > 0
                    else np.nan
                ),
                "median_ic": (
                    np.median(
                        ic_values
                    )
                    if len(ic_values) > 0
                    else np.nan
                ),
                "ic_signals":
                    len(ic_values)
            }
        )


    return pd.DataFrame(
        results
    )

In [14]:
# 06c-14. fold 1 최종 validation

xgb_final_validation_results = (
    evaluate_final_xgb_params(
        train_df,
        validation_df
    )
)


print(
    xgb_final_validation_results
    .sort_values(
        [
            "rmse",
            "mean_ic"
        ],
        ascending=[
            True,
            False
        ]
    )
    .head(20)
    .to_string(
        index=False
    )
)

 colsample_bytree  learning_rate  max_depth  min_child_weight  reg_lambda  subsample  reg_alpha  best_iteration     rmse      mae       r2   mean_ic  median_ic  ic_signals
              0.8           0.05          4                 0         1.0        1.0        0.0              74 0.064665 0.044015 0.026029 -0.018969  -0.008378          25
              0.8           0.05          4                 1         1.0        1.0        0.0              74 0.064665 0.044015 0.026029 -0.018969  -0.008378          25
              0.6           0.05          4                 0         0.1        1.0        0.0             132 0.064681 0.043942 0.025554  0.025104  -0.025372          25
              0.6           0.05          4                 1         0.1        1.0        0.0             132 0.064681 0.043942 0.025554  0.025104  -0.025372          25
              0.4           0.03          4                 0         0.1        1.0        0.0             135 0.064715 0.043721 0.024538 -

수익률 값의 평균적인 오차는 가장 낮아졌지만, cross-sectional ranking 능력은 아직 약하다

In [15]:
# 06c-15. colsample 1.0 후보 보완

XGB_COLSAMPLE_1_CANDIDATES = list(
    ParameterGrid(
        {
            "max_depth": [
                3,
                4,
                5
            ],
            "learning_rate": [
                0.01,
                0.03,
                0.05
            ],
            "min_child_weight": [
                0,
                1,
                5
            ],
            "colsample_bytree": [
                1.0
            ],
            "reg_lambda": [
                0.1,
                1.0,
                10.0
            ]
        }
    )
)

print(
    "extra candidates:",
    len(XGB_COLSAMPLE_1_CANDIDATES)
)

extra candidates: 81


In [16]:
# 06c-16. colsample 1.0 validation

def evaluate_xgb_candidate_list(
    train_df,
    validation_df,
    candidates
):

    X_train = train_df[
        MODEL_FEATURES
    ]

    y_train = train_df[
        TARGET
    ]

    X_validation = validation_df[
        MODEL_FEATURES
    ]

    y_validation = validation_df[
        TARGET
    ]

    results = []


    for params in candidates:

        model = XGBRegressor(
            n_estimators=3000,
            max_depth=params[
                "max_depth"
            ],
            learning_rate=params[
                "learning_rate"
            ],
            min_child_weight=params[
                "min_child_weight"
            ],
            subsample=1.0,
            colsample_bytree=params[
                "colsample_bytree"
            ],
            reg_alpha=0.0,
            reg_lambda=params[
                "reg_lambda"
            ],
            objective="reg:squarederror",
            eval_metric="rmse",
            early_stopping_rounds=100,
            tree_method="hist",
            random_state=42,
            n_jobs=-1
        )

        model.fit(
            X_train,
            y_train,
            eval_set=[
                (
                    X_validation,
                    y_validation
                )
            ],
            verbose=False
        )

        prediction = model.predict(
            X_validation
        )

        result_df = (
            validation_df[
                [
                    "signal_date",
                    "ticker",
                    TARGET
                ]
            ]
            .copy()
        )

        result_df[
            "prediction"
        ] = prediction

        rmse = np.sqrt(
            mean_squared_error(
                y_validation,
                prediction
            )
        )

        mae = mean_absolute_error(
            y_validation,
            prediction
        )

        r2 = r2_score(
            y_validation,
            prediction
        )

        ic_values = calculate_ic(
            result_df,
            "prediction",
            TARGET
        )

        results.append(
            {
                **params,
                "subsample": 1.0,
                "reg_alpha": 0.0,
                "best_iteration":
                    model.best_iteration,
                "rmse": rmse,
                "mae": mae,
                "r2": r2,
                "mean_ic": (
                    ic_values.mean()
                    if len(ic_values) > 0
                    else np.nan
                ),
                "median_ic": (
                    np.median(
                        ic_values
                    )
                    if len(ic_values) > 0
                    else np.nan
                ),
                "ic_signals":
                    len(ic_values)
            }
        )

    return pd.DataFrame(
        results
    )

In [17]:
# 06c-17. 최종 xgboost validation 결과

xgb_colsample_1_results = (
    evaluate_xgb_candidate_list(
        train_df,
        validation_df,
        XGB_COLSAMPLE_1_CANDIDATES
    )
)

xgb_all_validation_results = (
    pd.concat(
        [
            xgb_final_validation_results,
            xgb_colsample_1_results
        ],
        ignore_index=True
    )
)

print(
    xgb_all_validation_results
    .sort_values(
        [
            "rmse",
            "mean_ic"
        ],
        ascending=[
            True,
            False
        ]
    )
    .head(20)
    .to_string(
        index=False
    )
)

 colsample_bytree  learning_rate  max_depth  min_child_weight  reg_lambda  subsample  reg_alpha  best_iteration     rmse      mae       r2   mean_ic  median_ic  ic_signals
              0.8           0.05          4                 0         1.0        1.0        0.0              74 0.064665 0.044015 0.026029 -0.018969  -0.008378          25
              0.8           0.05          4                 1         1.0        1.0        0.0              74 0.064665 0.044015 0.026029 -0.018969  -0.008378          25
              0.6           0.05          4                 0         0.1        1.0        0.0             132 0.064681 0.043942 0.025554  0.025104  -0.025372          25
              0.6           0.05          4                 1         0.1        1.0        0.0             132 0.064681 0.043942 0.025554  0.025104  -0.025372          25
              0.4           0.03          4                 0         0.1        1.0        0.0             135 0.064715 0.043721 0.024538 -

In [18]:
# 06c-18. xgboost parameter 최종 확정

best_xgb_row = (
    xgb_all_validation_results
    .sort_values(
        [
            "rmse",
            "mean_ic"
        ],
        ascending=[
            True,
            False
        ]
    )
    .iloc[0]
)

print(
    best_xgb_row
)

colsample_bytree     0.800000
learning_rate        0.050000
max_depth            4.000000
min_child_weight     0.000000
reg_lambda           1.000000
subsample            1.000000
reg_alpha            0.000000
best_iteration      74.000000
rmse                 0.064665
mae                  0.044015
r2                   0.026029
mean_ic             -0.018969
median_ic           -0.008378
ic_signals          25.000000
Name: 226, dtype: float64


현재 fold 1 validation의 최종 선택:

max_depth          = 4
learning_rate      = 0.05
min_child_weight   = 0
subsample          = 1.0
colsample_bytree   = 0.8
reg_alpha          = 0
reg_lambda         = 1

best_iteration     = 74
RMSE               = 0.064665
R²                 = +0.026029
mean IC            = -0.018969

Ridge fold 1 validation   RMSE ≈ 0.065537
RF fold 1 validation      RMSE ≈ 0.065137
XGB fold 1 validation     RMSE ≈ 0.064665

In [19]:
# 06c-19. xgboost 최종 후보 고정

XGB_ALL_CANDIDATES = (
    XGB_FINAL_CANDIDATES
    + XGB_COLSAMPLE_1_CANDIDATES
)

print(
    "final candidates:",
    len(XGB_ALL_CANDIDATES)
)

final candidates: 324


walk-forward는 각 시점에서:

그 시점까지의 train
        ↓
그 시점 validation
        ↓
best parameter 선택
        ↓
다음 6개월 test

구조니까 fold마다 최적값이 달라질 수 있음.

각 fold에서
324개 parameter candidate
× early stopping XGBoost
× 9 folds
를 돌리니까 총 2,916개의 XGBoost validation model을 학습

In [20]:
# 06c-20. xgboost walk-forward 실행

XGB_RESULT_DIR = (
    PROJECT_ROOT
    / "data"
    / "predictions"
    / "06c_xgboost"
)

XGB_RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


xgb_fold_rows = []
xgb_oos_frames = []


for fold in folds:

    print(
        "fold",
        fold["fold"],
        "start"
    )


    train_mask = (
        (
            supervised_dataset["signal_date"]
            < fold["train_end"]
        )
        &
        (
            supervised_dataset["next_execution_date"]
            <= fold["train_end"]
        )
    )

    validation_mask = (
        (
            supervised_dataset["signal_date"]
            >= fold["validation_start"]
        )
        &
        (
            supervised_dataset["signal_date"]
            < fold["validation_end"]
        )
        &
        (
            supervised_dataset["next_execution_date"]
            <= fold["validation_end"]
        )
    )

    test_mask = (
        (
            supervised_dataset["signal_date"]
            >= fold["test_start"]
        )
        &
        (
            supervised_dataset["signal_date"]
            < fold["test_end"]
        )
        &
        (
            supervised_dataset["next_execution_date"]
            <= fold["test_end"]
        )
    )


    train_df = (
        supervised_dataset.loc[
            train_mask
        ]
        .copy()
    )

    validation_df = (
        supervised_dataset.loc[
            validation_mask
        ]
        .copy()
    )

    test_df = (
        supervised_dataset.loc[
            test_mask
        ]
        .copy()
    )


    validation_results = (
        evaluate_xgb_candidate_list(
            train_df,
            validation_df,
            XGB_ALL_CANDIDATES
        )
    )


    best_row = (
        validation_results
        .sort_values(
            [
                "rmse",
                "mean_ic"
            ],
            ascending=[
                True,
                False
            ]
        )
        .iloc[0]
    )


    best_n_estimators = (
        int(
            best_row[
                "best_iteration"
            ]
        )
        + 1
    )


    train_validation_mask = (
        (
            supervised_dataset["signal_date"]
            < fold["test_start"]
        )
        &
        (
            supervised_dataset["next_execution_date"]
            <= fold["test_start"]
        )
    )


    train_validation_df = (
        supervised_dataset.loc[
            train_validation_mask
        ]
        .copy()
    )


    X_train_validation = (
        train_validation_df[
            MODEL_FEATURES
        ]
    )

    y_train_validation = (
        train_validation_df[
            TARGET
        ]
    )

    X_test = (
        test_df[
            MODEL_FEATURES
        ]
    )

    y_test = (
        test_df[
            TARGET
        ]
    )


    xgb_model = XGBRegressor(
        n_estimators=best_n_estimators,
        max_depth=int(
            best_row[
                "max_depth"
            ]
        ),
        learning_rate=float(
            best_row[
                "learning_rate"
            ]
        ),
        min_child_weight=float(
            best_row[
                "min_child_weight"
            ]
        ),
        subsample=float(
            best_row[
                "subsample"
            ]
        ),
        colsample_bytree=float(
            best_row[
                "colsample_bytree"
            ]
        ),
        reg_alpha=float(
            best_row[
                "reg_alpha"
            ]
        ),
        reg_lambda=float(
            best_row[
                "reg_lambda"
            ]
        ),
        objective="reg:squarederror",
        eval_metric="rmse",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )


    xgb_model.fit(
        X_train_validation,
        y_train_validation
    )


    xgb_pred = xgb_model.predict(
        X_test
    )


    dummy_model = DummyRegressor(
        strategy="mean"
    )

    dummy_model.fit(
        X_train_validation,
        y_train_validation
    )

    dummy_pred = dummy_model.predict(
        X_test
    )


    test_result = (
        test_df[
            [
                "signal_date",
                "execution_date",
                "ticker",
                "name",
                TARGET
            ]
        ]
        .copy()
    )

    test_result[
        "prediction"
    ] = xgb_pred

    test_result[
        "dummy_prediction"
    ] = dummy_pred

    test_result[
        "fold"
    ] = fold["fold"]


    ic_values = calculate_ic(
        test_result,
        "prediction",
        TARGET
    )


    xgb_rmse = np.sqrt(
        mean_squared_error(
            y_test,
            xgb_pred
        )
    )

    xgb_mae = mean_absolute_error(
        y_test,
        xgb_pred
    )

    xgb_r2 = r2_score(
        y_test,
        xgb_pred
    )


    dummy_rmse = np.sqrt(
        mean_squared_error(
            y_test,
            dummy_pred
        )
    )

    dummy_mae = mean_absolute_error(
        y_test,
        dummy_pred
    )

    dummy_r2 = r2_score(
        y_test,
        dummy_pred
    )


    xgb_fold_rows.append(
        {
            "fold": fold["fold"],
            "max_depth": int(
                best_row["max_depth"]
            ),
            "learning_rate": float(
                best_row["learning_rate"]
            ),
            "min_child_weight": float(
                best_row["min_child_weight"]
            ),
            "subsample": float(
                best_row["subsample"]
            ),
            "colsample_bytree": float(
                best_row["colsample_bytree"]
            ),
            "reg_alpha": float(
                best_row["reg_alpha"]
            ),
            "reg_lambda": float(
                best_row["reg_lambda"]
            ),
            "best_iteration": int(
                best_row["best_iteration"]
            ),
            "n_estimators":
                best_n_estimators,
            "validation_rmse": float(
                best_row["rmse"]
            ),
            "test_start":
                fold["test_start"],
            "test_end":
                fold["test_end"],
            "test_rows":
                len(test_df),
            "test_signals":
                test_df[
                    "signal_date"
                ].nunique(),
            "xgb_rmse":
                xgb_rmse,
            "xgb_mae":
                xgb_mae,
            "xgb_r2":
                xgb_r2,
            "mean_ic": (
                ic_values.mean()
                if len(ic_values) > 0
                else np.nan
            ),
            "median_ic": (
                np.median(
                    ic_values
                )
                if len(ic_values) > 0
                else np.nan
            ),
            "ic_signals":
                len(ic_values),
            "dummy_rmse":
                dummy_rmse,
            "dummy_mae":
                dummy_mae,
            "dummy_r2":
                dummy_r2
        }
    )


    xgb_oos_frames.append(
        test_result
    )


    # checkpoint
    pd.DataFrame(
        xgb_fold_rows
    ).to_csv(
        XGB_RESULT_DIR
        / "xgb_fold_checkpoint.csv",
        index=False,
        encoding="utf-8-sig"
    )

    checkpoint_predictions = (
        pd.concat(
            xgb_oos_frames,
            ignore_index=True
        )
    )

    pq.write_table(
        pa.Table.from_pandas(
            checkpoint_predictions,
            preserve_index=False
        ),
        XGB_RESULT_DIR
        / "xgb_oos_checkpoint.parquet",
        compression="snappy"
    )


    print(
        "fold",
        fold["fold"],
        "done"
    )

fold 1 start
fold 1 done
fold 2 start
fold 2 done
fold 3 start
fold 3 done
fold 4 start
fold 4 done
fold 5 start
fold 5 done
fold 6 start
fold 6 done
fold 7 start
fold 7 done
fold 8 start
fold 8 done
fold 9 start
fold 9 done


In [21]:
# 06c-21. xgboost fold 결과

xgb_fold_results = pd.DataFrame(
    xgb_fold_rows
)

xgb_fold_results[
    "rmse_improvement"
] = (
    xgb_fold_results[
        "dummy_rmse"
    ]
    - xgb_fold_results[
        "xgb_rmse"
    ]
)

xgb_fold_results[
    "xgb_better"
] = (
    xgb_fold_results[
        "xgb_rmse"
    ]
    <
    xgb_fold_results[
        "dummy_rmse"
    ]
)


print(
    xgb_fold_results.to_string(
        index=False
    )
)

 fold  max_depth  learning_rate  min_child_weight  subsample  colsample_bytree  reg_alpha  reg_lambda  best_iteration  n_estimators  validation_rmse test_start   test_end  test_rows  test_signals  xgb_rmse  xgb_mae    xgb_r2   mean_ic  median_ic  ic_signals  dummy_rmse  dummy_mae  dummy_r2  rmse_improvement  xgb_better
    1          4           0.05               0.0        1.0               0.8        0.0         1.0              74            75         0.064665 2021-11-04 2022-05-04       1250            25  0.064984 0.047729 -0.039140  0.054452   0.062622          25    0.064548   0.047621 -0.025234         -0.000436       False
    2          3           0.05               0.0        1.0               0.4        0.0         0.1              11            12         0.063958 2022-05-04 2022-11-04       1250            25  0.070949 0.051634 -0.204925 -0.080671  -0.059528          24    0.065276   0.047685 -0.019936         -0.005673       False
    3          4           0.03      

In [22]:
# 06c-22. xgboost oos prediction 결합

xgb_oos_predictions = (
    pd.concat(
        xgb_oos_frames,
        ignore_index=True
    )
    .sort_values(
        [
            "signal_date",
            "ticker"
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "shape:",
    xgb_oos_predictions.shape
)

print(
    "signals:",
    xgb_oos_predictions[
        "signal_date"
    ].nunique()
)

print(
    "start:",
    xgb_oos_predictions[
        "signal_date"
    ].min()
)

print(
    "end:",
    xgb_oos_predictions[
        "signal_date"
    ].max()
)

print(
    "duplicates:",
    xgb_oos_predictions[
        [
            "signal_date",
            "ticker"
        ]
    ]
    .duplicated()
    .sum()
)

shape: (11140, 8)
signals: 223
start: 2021-11-05 00:00:00
end: 2026-04-24 00:00:00
duplicates: 0


In [23]:
# 06c-23. xgboost 전체 oos 성능

pooled_xgb_rmse = np.sqrt(
    mean_squared_error(
        xgb_oos_predictions[
            TARGET
        ],
        xgb_oos_predictions[
            "prediction"
        ]
    )
)

pooled_xgb_mae = mean_absolute_error(
    xgb_oos_predictions[
        TARGET
    ],
    xgb_oos_predictions[
        "prediction"
    ]
)

pooled_xgb_r2 = r2_score(
    xgb_oos_predictions[
        TARGET
    ],
    xgb_oos_predictions[
        "prediction"
    ]
)

pooled_xgb_ic = calculate_ic(
    xgb_oos_predictions,
    "prediction",
    TARGET
)


print(
    "pooled rmse:",
    pooled_xgb_rmse
)

print(
    "pooled mae:",
    pooled_xgb_mae
)

print(
    "pooled r2:",
    pooled_xgb_r2
)

print(
    "mean ic:",
    pooled_xgb_ic.mean()
)

print(
    "median ic:",
    np.median(
        pooled_xgb_ic
    )
)

print(
    "ic signals:",
    len(
        pooled_xgb_ic
    )
)

print(
    "xgb better folds:",
    xgb_fold_results[
        "xgb_better"
    ].sum(),
    "/",
    len(
        xgb_fold_results
    )
)

pooled rmse: 0.0795071201622273
pooled mae: 0.05529956223471987
pooled r2: -0.023635177660239926
mean ic: -0.013658699527413063
median ic: -0.015127009347099204
ic signals: 150
xgb better folds: 3 / 9


In [24]:
# 06c-24. xgboost prediction 분산

xgb_prediction_stats = (
    xgb_oos_predictions
    .groupby(
        "signal_date"
    )
    .agg(
        prediction_std=(
            "prediction",
            "std"
        ),
        target_std=(
            TARGET,
            "std"
        )
    )
)


print(
    xgb_prediction_stats.describe()
)

       prediction_std  target_std
count      223.000000  223.000000
mean         0.003630    0.068816
std          0.005121    0.020857
min          0.000000    0.034853
25%          0.000000    0.051816
50%          0.000991    0.064346
75%          0.005732    0.082868
max          0.032481    0.137248


실제 다음 주 종목 차이
≈ 6.88%

XGBoost가 만들어낸 종목 차이
≈ 0.36%

전체 OOS signal = 223
IC 계산 가능     = 150
IC 계산 불가     = 73

어떤 기간
→ tree 1개면 충분

다른 기간
→ tree 250개 필요

현재 사용하고 있는 28개의 raw asset / market / macro feature로는 다음 주 개별 종목 수익률의 cross-sectional 차이를 안정적으로 예측하기 어렵고, 모델 복잡도를 높이는 것만으로는 OOS 성능이 개선되지 않았다.

In [25]:
# 06c-25. xgboost 결과 저장

XGB_FOLD_PATH = (
    XGB_RESULT_DIR
    / "xgb_fold_results.csv"
)

XGB_OOS_PATH = (
    XGB_RESULT_DIR
    / "xgb_oos_predictions.parquet"
)

XGB_PREDICTION_STATS_PATH = (
    XGB_RESULT_DIR
    / "xgb_prediction_stats.csv"
)


xgb_fold_results.to_csv(
    XGB_FOLD_PATH,
    index=False,
    encoding="utf-8-sig"
)

xgb_prediction_stats.to_csv(
    XGB_PREDICTION_STATS_PATH,
    encoding="utf-8-sig"
)

pq.write_table(
    pa.Table.from_pandas(
        xgb_oos_predictions,
        preserve_index=False
    ),
    XGB_OOS_PATH,
    compression="snappy"
)


print(
    "saved:",
    XGB_FOLD_PATH
)

print(
    "saved:",
    XGB_OOS_PATH
)

print(
    "saved:",
    XGB_PREDICTION_STATS_PATH
)

saved: C:\code\portfolio_optimization\data\predictions\06c_xgboost\xgb_fold_results.csv
saved: C:\code\portfolio_optimization\data\predictions\06c_xgboost\xgb_oos_predictions.parquet
saved: C:\code\portfolio_optimization\data\predictions\06c_xgboost\xgb_prediction_stats.csv


In [26]:
# 06c-26. 저장 확인

print(
    XGB_FOLD_PATH.exists(),
    XGB_OOS_PATH.exists(),
    XGB_PREDICTION_STATS_PATH.exists()
)

True True True
